# Statistics 2026-2 Final Grade Calculator

Public student instructions:

1. Download this notebook and `student_grade_template.csv` from the public course repository, or open the notebook in Colab.
2. Fill in `student_grade_template.csv` with your own grades. Keep the first column named `item`; put your grades in the second column.
3. Set `INPUT_FILE = Path("student_grade_template.csv")` near the top of the notebook.
4. Run the notebook from top to bottom.
5. Read the grade report printed at the end. It shows the final exam rule, Tutor/WooClap review calculation, assignment calculation, midterm magen comparison, and final course grade.

The same formulas are used for the staff gradebook calculation. The staff-only Moodle export files are not needed for student use.


## Controls

Set the input paths and output flags in the next cell. Then run the notebook from top to bottom.


In [1]:
from pathlib import Path
from zipfile import ZipFile
from xml.etree import ElementTree as ET
import math
import re

import pandas as pd

# Input can be either a CSV template or the Moodle gradebook XLSX export.
INPUT_FILE = Path("raw_exports/367143610120262 Grades.xlsx")
ATTENDANCE_FILE = Path("raw_exports/367143610120262_Attendances_20260724-1957.xlsx")
RESERVE_DUTY_FILE = Path("raw_exports/036714361courseList.xlsx")
POST_TEST_FILE = Path("raw_exports/moodle_grade_import (3).csv")

# Output controls.
WRITE_GRADE_REPORT_MARKDOWN = True
WRITE_MOODLE_IMPORT_CSV = True
WRITE_DETAILED_AUDIT_CSV = True

OUTPUT_DIR = Path("generated_outputs")
GRADE_REPORT_MARKDOWN = OUTPUT_DIR / "final_course_grade_report.md"
MOODLE_IMPORT_CSV = OUTPUT_DIR / "final_course_grade_moodle_import.csv"
DETAILED_AUDIT_CSV = OUTPUT_DIR / "final_course_grade_audit.csv"


## Policy Constants


In [2]:
IDENTITY_COLUMNS = ["Last name", "First name", "ID number", "Email address"]

FIRST_REVIEW_SESSION = 1
LAST_REVIEW_SESSION = 12
REVIEW_SESSIONS = range(FIRST_REVIEW_SESSION, LAST_REVIEW_SESSION + 1)

PERCENT_DENOMINATOR = 100.0
COURSE_GRADE_CAP = 100.0
FINAL_EXAM_PASS_THRESHOLD = 56.0

REVIEW_RECORDED_ACTIVITY_CAP = 7.2
REVIEW_ACTIVITY_BONUS = 0.6
REVIEW_ACTIVITY_CAP = REVIEW_RECORDED_ACTIVITY_CAP + REVIEW_ACTIVITY_BONUS

ATTENDED_ON_TIME_SESSION_MAX = 0.6
ATTENDED_LATE_SESSION_MAX = 0.5
EXCUSED_SESSION_MAX = 0.6
UNEXCUSED_ON_TIME_SESSION_MAX = 0.3
UNEXCUSED_LATE_SESSION_MAX = 0.1
RESERVE_EXCUSED_CATEGORIES = {2, 3}

NUMBER_OF_ASSIGNMENTS = 3
ASSIGNMENT_POINTS_IF_QUIZ_PASSED = 5.0
ASSIGNMENT_POINTS_IF_QUIZ_NOT_PASSED = 3.0
ASSIGNMENTS_TOTAL_POINTS = 15.0

MIDTERM_POINTS = 10.0
FINAL_EXAM_POINTS = 70.0
COURSE_POINTS_WITH_MIDTERM = 100.0
COURSE_POINTS_WITHOUT_MIDTERM = 90.0

ZEROED_CATEGORY_TOTALS = [
    "WooClap total",
    "Tutor Deadline total",
    "Tutor total",
    "Ungraded total",
]

WOOCLAP_COLUMNS_BY_SESSION = {
    1: "Wooclap: In class 1A (Real)",
    2: "Wooclap: In class 2 (Real)",
    3: "Wooclap: In class 3 (Real)",
    4: "Wooclap: In class 4 (Real)",
    5: "Wooclap: In class 5 (Real)",
    6: "Wooclap: In class 6 (Real)",
    7: "Wooclap: In class 7 (Real)",
    8: "Wooclap: In class 8 (Real)",
    9: None,
    10: "Wooclap: In class 10 (Real)",
    11: "Wooclap: In class 11 (Real)",
    12: None,
}


## XLSX Reader

This avoids extra dependencies such as `openpyxl`, which keeps the notebook easier to run in Colab.


In [3]:
def column_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    index = 0
    for letter in letters:
        index = index * 26 + ord(letter.upper()) - ord("A") + 1
    return index - 1


def read_xlsx_sheet(path, sheet_name):
    ns = {
        "main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "rel": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    }
    with ZipFile(path) as workbook:
        strings = []
        if "xl/sharedStrings.xml" in workbook.namelist():
            root = ET.fromstring(workbook.read("xl/sharedStrings.xml"))
            for item in root.findall("main:si", ns):
                strings.append("".join(item.itertext()))

        wb = ET.fromstring(workbook.read("xl/workbook.xml"))
        rels = ET.fromstring(workbook.read("xl/_rels/workbook.xml.rels"))
        relmap = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels}

        target = None
        for sheet in wb.findall(".//main:sheet", ns):
            if sheet.attrib["name"] == sheet_name:
                rel_id = sheet.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
                target = relmap[rel_id].lstrip("/")
                break
        if target is None:
            raise ValueError(f"Sheet {sheet_name!r} not found in {path}")
        if not target.startswith("xl/"):
            target = f"xl/{target}"

        sheet_xml = ET.fromstring(workbook.read(target))
        rows = []
        for row in sheet_xml.findall(".//main:row", ns):
            values = []
            for cell in row.findall("main:c", ns):
                value_node = cell.find("main:v", ns)
                value = ""
                if value_node is not None:
                    value = value_node.text or ""
                    if cell.attrib.get("t") == "s":
                        value = strings[int(value)]
                index = column_index(cell.attrib["r"])
                while len(values) <= index:
                    values.append("")
                values[index] = value
            if any(value != "" for value in values):
                rows.append(values)

    width = max(len(row) for row in rows)
    normalized_rows = [row + [""] * (width - len(row)) for row in rows]
    header = normalized_rows[0]
    if len(set(header)) != len(header) or "" in header:
        header = [f"column_{index + 1}" for index in range(width)]
        return pd.DataFrame(normalized_rows, columns=header)
    return pd.DataFrame(normalized_rows[1:], columns=header)


def read_attendance(path):
    raw = read_xlsx_sheet(path, "Attendances")
    header_row = raw.index[raw.iloc[:, 0] == "Last name"][0]
    headers = raw.iloc[header_row].tolist()
    data = raw.iloc[header_row + 1 :].copy()
    data.columns = headers
    data = data[data["Email address"].astype(str).str.contains("@", na=False)].copy()
    return data


## Shared Helpers


In [4]:
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")


def clean_number(value, default=0.0):
    if pd.isna(value):
        return default
    text = str(value).strip()
    if text in {"", "-"}:
        return default
    return float(text)


def has_number(value):
    if pd.isna(value):
        return False
    text = str(value).strip()
    if text in {"", "-"}:
        return False
    try:
        float(text)
        return True
    except ValueError:
        return False


def clean_student_id(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    return text


def as_bool(value):
    text = clean_text(value)
    if text in {"yes", "y", "true", "1", "passed", "pass"}:
        return True
    if text in {"no", "n", "false", "0", "failed", "fail", ""}:
        return False
    raise ValueError(f"Cannot interpret {value!r} as yes/no")


def attendance_status(value):
    text = str(value).strip().upper()
    if text.startswith("P"):
        return "attended"
    if text.startswith("E"):
        return "excused"
    return "unexcused"


def attendance_session_columns(attendance):
    date_pattern = re.compile(r"\d{1,2} \w{3} 2026")
    return [column for column in attendance.columns if date_pattern.search(str(column))]


def get(row, key, default=""):
    return row[key] if key in row.index else default


## Normalize CSV Or XLSX Input

After this step, both formats have the same internal row structure.


In [5]:
def read_reserve_categories(path):
    if not path.exists():
        return {}
    reserve = read_xlsx_sheet(path, "ציונים1")
    categories = {}
    for _, row in reserve.iterrows():
        email = str(row.get("אימייל", "")).strip()
        if not email:
            continue
        value_text = str(row.get("קבוצת מילואים", "")).strip()
        if value_text not in {"", "-"}:
            categories[email] = int(float(value_text))
    return categories


def read_post_test_scores(path):
    if not path.exists():
        return {}
    post_tests = pd.read_csv(path)
    if "ID number" not in post_tests.columns:
        raise ValueError(f"Post-test import is missing ID number column: {path}")
    scores_by_id = {}
    for _, source in post_tests.iterrows():
        student_id = clean_student_id(source.get("ID number", ""))
        if not student_id:
            continue
        scores = {}
        for session in REVIEW_SESSIONS:
            column = f"lecture_{session:02d}"
            if column in post_tests.columns and has_number(source.get(column, "")):
                scores[session] = clean_number(source[column])
        if scores:
            scores_by_id[student_id] = scores
    return scores_by_id


def normalize_csv_input(path):
    raw = pd.read_csv(path).set_index("item")
    rows = []
    for student in raw.columns:
        source = raw[student]
        row = pd.Series(dtype=object)
        row["source_row_index"] = len(rows)
        row["student"] = student
        row["Last name"] = ""
        row["First name"] = student
        row["ID number"] = ""
        row["Email address"] = ""
        reserve_category = clean_number(get(source, "Reserve category", get(source, "reserve_category", 0)))
        row["reserve_category"] = int(reserve_category)
        reserve_excused = row["reserve_category"] in RESERVE_EXCUSED_CATEGORIES
        for session in REVIEW_SESSIONS:
            prefix = f"session_{session:02d}"
            status = get(source, f"{prefix}_status", "unexcused")
            row[f"session_{session:02d}_status"] = "excused" if reserve_excused else status
            row[f"session_{session:02d}_wooclap"] = get(source, f"{prefix}_wooclap", 0)
            row[f"session_{session:02d}_tutor"] = get(source, f"{prefix}_tutor", 0)
            row[f"session_{session:02d}_tutor_timing"] = get(source, f"{prefix}_tutor_timing", "late")
            row[f"session_{session:02d}_post_test"] = get(source, f"{prefix}_post_test", "")
        for number in range(1, NUMBER_OF_ASSIGNMENTS + 1):
            row[f"assignment_{number}_grade"] = get(source, f"assignment_{number}_grade", 0)
            row[f"assignment_{number}_quiz_passed"] = get(source, f"assignment_{number}_quiz_passed", "no")
        row["midterm_grade"] = get(source, "midterm_grade", 0)
        row["final_exam_grade"] = get(source, "final_exam_grade", "")
        rows.append(row)
    return pd.DataFrame(rows), None


def normalize_xlsx_input(gradebook_path, attendance_path):
    gradebook = read_xlsx_sheet(gradebook_path, "Grades")
    attendance = read_attendance(attendance_path)
    reserve_categories = read_reserve_categories(RESERVE_DUTY_FILE)
    post_test_scores = read_post_test_scores(POST_TEST_FILE)
    attendance_by_email = attendance.set_index("Email address", drop=False)
    session_columns = attendance_session_columns(attendance)
    rows = []

    for source_index, source in gradebook.iterrows():
        row = pd.Series(dtype=object)
        row["source_row_index"] = source_index
        for column in IDENTITY_COLUMNS:
            row[column] = source[column]
        email = source["Email address"]
        row["student"] = email
        row["reserve_category"] = reserve_categories.get(email, 0)
        reserve_excused = row["reserve_category"] in RESERVE_EXCUSED_CATEGORIES
        attendance_row = attendance_by_email.loc[email] if email in attendance_by_email.index else None

        for session in REVIEW_SESSIONS:
            status = "unexcused"
            if attendance_row is not None and session <= len(session_columns):
                status = attendance_status(attendance_row[session_columns[session - 1]])
            if reserve_excused:
                status = "excused"
            wooclap_column = WOOCLAP_COLUMNS_BY_SESSION.get(session)
            row[f"session_{session:02d}_status"] = status
            row[f"session_{session:02d}_wooclap"] = clean_number(source[wooclap_column]) if wooclap_column else 0
            row[f"session_{session:02d}_tutor"] = clean_number(source.get(f"Assignment: Tutor {session} (Real)", 0))
            row[f"session_{session:02d}_tutor_timing"] = "on_time" if clean_number(source.get(f"Tutor {session} on time (Real)", 0)) > 0 else "late"
            row[f"session_{session:02d}_post_test"] = post_test_scores.get(clean_student_id(source["ID number"]), {}).get(session, "")

        for number in range(1, NUMBER_OF_ASSIGNMENTS + 1):
            row[f"assignment_{number}_grade"] = clean_number(source.get(f"Assignment: Assignment {number} (Real)", 0))
            regular_quiz = clean_number(source.get(f"Quiz: Assignment {number} Quiz (Real)", 0))
            makeup_quiz = clean_number(source.get(f"Quiz: Make-Up Assignment {number} Quiz (Real)", 0))
            row[f"assignment_{number}_quiz_passed"] = regular_quiz > 0 or makeup_quiz > 0
        row["midterm_grade"] = clean_number(source.get("Midterm total (Real)", 0))
        row["final_exam_grade"] = source.get("Final exam total (Real)", "")
        rows.append(row)
    return pd.DataFrame(rows), gradebook


def load_input(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        normalized, original = normalize_csv_input(path)
        return normalized, original, "csv"
    if suffix == ".xlsx":
        normalized, original = normalize_xlsx_input(path, ATTENDANCE_FILE)
        return normalized, original, "xlsx"
    raise ValueError(f"Unsupported input format: {path}")

students, original_gradebook, input_format = load_input(INPUT_FILE)
print(f"Loaded {len(students)} row(s) from {INPUT_FILE.name} as {input_format}.")
students.head()


Loaded 65 row(s) from 367143610120262 Grades.xlsx as xlsx.


,source_row_index,Last name,First name,ID number,Email address,student,reserve_category,session_01_status,session_01_wooclap,session_01_tutor,...,session_12_tutor_timing,session_12_post_test,assignment_1_grade,assignment_1_quiz_passed,assignment_2_grade,assignment_2_quiz_passed,assignment_3_grade,assignment_3_quiz_passed,midterm_grade,final_exam_grade
0,0,אדמוני,מאיה,322818386,mayaadm@post.bgu.ac.il,mayaadm@post.bgu.ac.il,1,excused,81.82,86.0,...,on_time,,99.0,True,100.0,True,100.0,True,66.51,68.22
1,1,אדר,נוי,323060202,noyad@post.bgu.ac.il,noyad@post.bgu.ac.il,0,excused,0.00,0.0,...,late,,94.0,True,93.0,True,100.0,True,88.00,75.43
2,2,אילביץ,אילנה,322772187,ilevich@post.bgu.ac.il,ilevich@post.bgu.ac.il,3,excused,0.00,95.0,...,on_time,,91.0,True,76.0,True,92.0,True,65.01,47.52
3,3,אלחואגרה,מחמד,325332419,alhwamoh@post.bgu.ac.il,alhwamoh@post.bgu.ac.il,0,excused,90.91,0.0,...,late,,92.0,True,97.0,True,83.0,True,71.52,57.94
4,4,אלקלעי,עדן,211340161,edenalk@post.bgu.ac.il,edenalk@post.bgu.ac.il,0,excused,100.00,83.0,...,on_time,,99.0,True,97.0,True,99.0,True,90.02,83.9


## Shared Calculation Functions


In [6]:
def session_max_points(status, timing):
    if status == "attended":
        return ATTENDED_ON_TIME_SESSION_MAX if timing == "on_time" else ATTENDED_LATE_SESSION_MAX
    if status == "excused":
        return EXCUSED_SESSION_MAX
    if status == "unexcused":
        return UNEXCUSED_ON_TIME_SESSION_MAX if timing == "on_time" else UNEXCUSED_LATE_SESSION_MAX
    raise ValueError(f"Unknown session status {status!r}")


def combined_review_percent(wooclap_grade, tutor_grade):
    return wooclap_grade + (PERCENT_DENOMINATOR - wooclap_grade) * tutor_grade / PERCENT_DENOMINATOR


def apply_post_test_bonus(percent_grade, post_test_score):
    if not has_number(post_test_score):
        return percent_grade, 0.0, False
    adjusted = percent_grade + (PERCENT_DENOMINATOR - percent_grade) * 0.5
    return adjusted, adjusted - percent_grade, True


def calculate_review(row):
    lines = []
    total = 0.0
    post_test_sessions = []
    post_test_gain_points = 0.0
    reserve_note = " reserve category " + str(row["reserve_category"]) if row.get("reserve_category", 0) in RESERVE_EXCUSED_CATEGORIES else ""
    for session in REVIEW_SESSIONS:
        status = clean_text(get(row, f"session_{session:02d}_status", "unexcused"))
        timing = clean_text(get(row, f"session_{session:02d}_tutor_timing", "late"))
        wooclap_grade = clean_number(get(row, f"session_{session:02d}_wooclap", 0))
        tutor_grade = clean_number(get(row, f"session_{session:02d}_tutor", 0))
        percent_grade0 = combined_review_percent(wooclap_grade, tutor_grade)
        post_test_score = get(row, f"session_{session:02d}_post_test", "")
        combined_percent, post_test_gain_percent, used_post_test = apply_post_test_bonus(percent_grade0, post_test_score)
        max_points = session_max_points(status, timing)
        session_total = combined_percent / PERCENT_DENOMINATOR * max_points
        post_test_gain_points += post_test_gain_percent / PERCENT_DENOMINATOR * max_points
        total += session_total
        post_test_note = ""
        if used_post_test:
            post_test_sessions.append(f"S{session:02d}")
            post_test_note = (
                f", post-test {clean_number(post_test_score):.1f}% -> "
                f"adjusted {combined_percent:.1f}% (+{post_test_gain_percent:.1f})"
            )
        lines.append(
            f"S{session:02d}: {status}{reserve_note}, WooClap {wooclap_grade:.1f}%, "
            f"Tutor {tutor_grade:.1f}%/{timing}, combined {percent_grade0:.1f}%{post_test_note}, "
            f"max {max_points:.2f}, total {session_total:.2f}"
        )

    review_activity_points = total + REVIEW_ACTIVITY_BONUS
    feedback = (
        f"Tutor and WooClap review activity: {review_activity_points:.2f}/{REVIEW_ACTIVITY_CAP:.1f} "
        f"(raw {total:.2f}, bonus {REVIEW_ACTIVITY_BONUS:.1f}).\n"
        + "\n".join(lines)
    )
    return {
        "review_activity_points": review_activity_points,
        "review_activity_raw_points": total,
        "review_activity_bonus": REVIEW_ACTIVITY_BONUS,
        "review_post_test_sessions": ", ".join(post_test_sessions),
        "review_post_test_gain_points": post_test_gain_points,
        "review_feedback": feedback,
    }

def calculate_assignments(row):
    points = []
    lines = []
    for number in range(1, NUMBER_OF_ASSIGNMENTS + 1):
        grade = clean_number(get(row, f"assignment_{number}_grade", 0))
        passed = as_bool(get(row, f"assignment_{number}_quiz_passed", "no"))
        max_points = ASSIGNMENT_POINTS_IF_QUIZ_PASSED if passed else ASSIGNMENT_POINTS_IF_QUIZ_NOT_PASSED
        assignment_points = max_points * grade / PERCENT_DENOMINATOR
        points.append(assignment_points)
        rule = "5-point maximum" if passed else "scaled to 3-point maximum"
        lines.append(f"A{number}: grade {grade:.1f}%, quiz passed {passed}, {rule}, points {assignment_points:.2f}/5")
    total = sum(points)
    return {
        "assignments_total_points": total,
        "assignments_feedback": f"Assignments total: {total:.2f}/{ASSIGNMENTS_TOTAL_POINTS:.0f}.\n" + "\n".join(lines),
    }


def calculate_course(row, review, assignments):
    if not has_number(get(row, "final_exam_grade", "")):
        return {"skip": True}
    final_exam_grade = clean_number(get(row, "final_exam_grade"))
    midterm_grade = clean_number(get(row, "midterm_grade", 0))
    final_exam_points = FINAL_EXAM_POINTS * final_exam_grade / PERCENT_DENOMINATOR
    midterm_points = MIDTERM_POINTS * midterm_grade / PERCENT_DENOMINATOR

    if final_exam_grade < FINAL_EXAM_PASS_THRESHOLD:
        return {
            "skip": False,
            "final_exam_grade": final_exam_grade,
            "final_exam_points": final_exam_points,
            "midterm_points": midterm_points,
            "midterm_used": False,
            "grade_with_midterm": math.nan,
            "grade_without_midterm": math.nan,
            "grade_before_cap": final_exam_grade,
            "final_course_grade": final_exam_grade,
            "course_feedback": (
                f"Final course grade: {final_exam_grade:.2f}.\n"
                f"Moed A exam grade {final_exam_grade:.2f} is below the pass threshold of {FINAL_EXAM_PASS_THRESHOLD:.0f}, "
                f"so the final course grade is the exam grade.\nOther course components were not added."
            ),
            "diagnostic_note": "Final exam below pass threshold; final grade is exam grade.",
        }

    grade_with_midterm = review["review_activity_points"] + assignments["assignments_total_points"] + midterm_points + final_exam_points
    grade_without_midterm = (review["review_activity_points"] + assignments["assignments_total_points"] + final_exam_points) / COURSE_POINTS_WITHOUT_MIDTERM * COURSE_POINTS_WITH_MIDTERM
    grade_before_cap = max(grade_with_midterm, grade_without_midterm)
    final_grade = min(grade_before_cap, COURSE_GRADE_CAP)
    midterm_used = grade_with_midterm >= grade_without_midterm
    feedback = (
        f"Final course grade: {final_grade:.2f}.\n"
        f"Final exam: {final_exam_grade:.2f}; pass threshold {FINAL_EXAM_PASS_THRESHOLD:.0f}; passed.\n"
        f"Review activity: {review['review_activity_points']:.2f}/{REVIEW_ACTIVITY_CAP:.1f}.\n"
        f"Assignments: {assignments['assignments_total_points']:.2f}/{ASSIGNMENTS_TOTAL_POINTS:.0f}.\n"
        f"Midterm contribution: {midterm_points:.2f}/{MIDTERM_POINTS:.0f}; midterm used: {midterm_used}.\n"
        f"Grade with midterm: {grade_with_midterm:.2f}.\n"
        f"Grade without midterm: {grade_without_midterm:.2f}.\n"
        f"Grade before cap: {grade_before_cap:.2f}; cap {COURSE_GRADE_CAP:.0f}."
    )
    return {
        "skip": False,
        "final_exam_grade": final_exam_grade,
        "final_exam_points": final_exam_points,
        "midterm_points": midterm_points,
        "midterm_used": midterm_used,
        "grade_with_midterm": grade_with_midterm,
        "grade_without_midterm": grade_without_midterm,
        "grade_before_cap": grade_before_cap,
        "final_course_grade": final_grade,
        "course_feedback": feedback,
        "diagnostic_note": "OK",
    }


def calculate_student(row):
    review = calculate_review(row)
    assignments = calculate_assignments(row)
    course = calculate_course(row, review, assignments)
    return review, assignments, course


## Validate Inputs

Validation is separate from calculation. If these checks pass, the formula cells below can stay simple and readable.


In [7]:
VALID_STATUSES = {"attended", "excused", "unexcused"}
VALID_TIMINGS = {"on_time", "late"}


def validation_number(value):
    if pd.isna(value) or str(value).strip() in {"", "-"}:
        return None
    try:
        return float(value)
    except ValueError:
        return None


def validate_percent(value, label, issues, required=False):
    number = validation_number(value)
    if number is None:
        if required:
            issues.append(f"{label}: missing or non-numeric value {value!r}")
        return
    if number < 0 or number > PERCENT_DENOMINATOR:
        issues.append(f"{label}: expected 0-100, found {number}")


def validate_inputs(students):
    issues = []
    warnings = []

    for row_number, row in students.iterrows():
        label = row.get("student", row.get("Email address", f"row {row_number}"))
        has_final_exam = has_number(get(row, "final_exam_grade", ""))

        reserve_category = validation_number(get(row, "reserve_category", 0))
        if reserve_category is None or reserve_category < 0:
            issues.append(f"{label} reserve_category: expected a non-negative number, found {get(row, 'reserve_category', '')!r}")

        validate_percent(get(row, "final_exam_grade", ""), f"{label} final_exam_grade", issues, required=False)
        validate_percent(get(row, "midterm_grade", 0), f"{label} midterm_grade", issues)

        if has_final_exam:
            for number in range(1, NUMBER_OF_ASSIGNMENTS + 1):
                validate_percent(get(row, f"assignment_{number}_grade", 0), f"{label} assignment_{number}_grade", issues, required=True)
                try:
                    as_bool(get(row, f"assignment_{number}_quiz_passed", "no"))
                except ValueError as error:
                    issues.append(f"{label} assignment_{number}_quiz_passed: {error}")

        for session in REVIEW_SESSIONS:
            status = clean_text(get(row, f"session_{session:02d}_status", "unexcused"))
            timing = clean_text(get(row, f"session_{session:02d}_tutor_timing", "late"))
            if status not in VALID_STATUSES:
                issues.append(f"{label} session_{session:02d}_status: expected {sorted(VALID_STATUSES)}, found {status!r}")
            if timing not in VALID_TIMINGS:
                issues.append(f"{label} session_{session:02d}_tutor_timing: expected {sorted(VALID_TIMINGS)}, found {timing!r}")
            validate_percent(get(row, f"session_{session:02d}_wooclap", 0), f"{label} session_{session:02d}_wooclap", issues)
            validate_percent(get(row, f"session_{session:02d}_tutor", 0), f"{label} session_{session:02d}_tutor", issues)
            validate_percent(get(row, f"session_{session:02d}_post_test", ""), f"{label} session_{session:02d}_post_test", issues)

        if not issues:
            review_check = calculate_review(row)
            if review_check["review_activity_points"] > REVIEW_ACTIVITY_CAP + 1e-9:
                issues.append(
                    f"{label} review activity exceeds {REVIEW_ACTIVITY_CAP:.1f}; "
                    f"found {review_check['review_activity_points']:.2f}"
                )

    if input_format == "xlsx" and original_gradebook is not None:
        missing_targets = []
        for total in ZEROED_CATEGORY_TOTALS:
            missing_targets.extend([column for column in [f"{total} (Real)", f"{total} (Feedback)"] if column not in original_gradebook.columns])
        required_targets = [
            "Tutor and WooClap total (Real)",
            "Tutor and WooClap total (Feedback)",
            "Assignments total (Real)",
            "Assignments total (Feedback)",
            "Final exam total (Real)",
            "Final exam total (Feedback)",
            "Course total (Real)",
            "Course total (Feedback)",
        ]
        missing_targets.extend([column for column in required_targets if column not in original_gradebook.columns])
        if missing_targets:
            issues.append("Missing upload target columns: " + ", ".join(missing_targets))

    return issues, warnings


validation_issues, validation_warnings = validate_inputs(students)
print(f"Validation issues: {len(validation_issues)}")
for issue in validation_issues[:20]:
    print("-", issue)
if len(validation_issues) > 20:
    print(f"... {len(validation_issues) - 20} more issue(s)")

if validation_issues:
    raise ValueError("Input validation failed. Fix the issues above before calculating grades.")


Validation issues: 0


## Calculate Results


In [8]:
upload_rows = []
audit_rows = []
markdown_sections = []
skipped = []

for _, row in students.iterrows():
    review, assignments, course = calculate_student(row)
    if course["skip"]:
        skipped.append(row.get("student", row.get("Email address", "unknown")))
        continue

    identity = {column: row.get(column, "") for column in IDENTITY_COLUMNS}
    label = row.get("student", row.get("Email address", row.get("First name", "student")))

    upload = identity.copy()
    for total in ZEROED_CATEGORY_TOTALS:
        upload[f"{total} (Real)"] = 0
        upload[f"{total} (Feedback)"] = "Zeroed; see Tutor and WooClap total or Course total audit."
    upload["Tutor and WooClap total (Real)"] = round(review["review_activity_points"], 2)
    upload["Tutor and WooClap total (Feedback)"] = review["review_feedback"]
    upload["Assignments total (Real)"] = round(assignments["assignments_total_points"], 2)
    upload["Assignments total (Feedback)"] = assignments["assignments_feedback"]
    upload["Final exam total (Real)"] = round(course["final_exam_grade"], 2)
    upload["Final exam total (Feedback)"] = f"Moed A final exam grade {course['final_exam_grade']:.2f}; pass threshold {FINAL_EXAM_PASS_THRESHOLD:.0f}."
    upload["Course total (Real)"] = round(course["final_course_grade"], 2)
    upload["Course total (Feedback)"] = course["course_feedback"]
    upload_rows.append(upload)

    audit = identity.copy()
    audit.update({
        "student": label,
        "reserve_category": row.get("reserve_category", 0),
        "review_activity_points": review["review_activity_points"],
        "review_activity_raw_points": review["review_activity_raw_points"],
        "review_activity_bonus": review["review_activity_bonus"],
        "review_post_test_sessions": review["review_post_test_sessions"],
        "review_post_test_gain_points": review["review_post_test_gain_points"],
        "assignments_total_points": assignments["assignments_total_points"],
        "final_exam_grade": course["final_exam_grade"],
        "final_exam_points": course["final_exam_points"],
        "midterm_points": course["midterm_points"],
        "midterm_used": course["midterm_used"],
        "grade_with_midterm": course["grade_with_midterm"],
        "grade_without_midterm": course["grade_without_midterm"],
        "grade_before_cap": course["grade_before_cap"],
        "final_course_grade": course["final_course_grade"],
        "diagnostic_note": course["diagnostic_note"],
    })
    audit_rows.append(audit)

    markdown_sections.append(
        f"## {label}\n\n"
        f"### Course total\n\n{course['course_feedback']}\n\n"
        f"### Tutor and WooClap\n\n{review['review_feedback']}\n\n"
        f"### Assignments\n\n{assignments['assignments_feedback']}\n"
    )

upload_df = pd.DataFrame(upload_rows)
audit_df = pd.DataFrame(audit_rows)

print(f"Calculated grades for {len(audit_df)} row(s).")
if skipped:
    print(f"Skipped {len(skipped)} row(s) without a final exam grade.")
print(f"Upload CSV shape: {upload_df.shape}")
audit_df.head()


Calculated grades for 49 row(s).
Skipped 16 row(s) without a final exam grade.
Upload CSV shape: (49, 20)


,Last name,First name,ID number,Email address,student,reserve_category,review_activity_points,review_activity_raw_points,review_activity_bonus,review_post_test_sessions,...,assignments_total_points,final_exam_grade,final_exam_points,midterm_points,midterm_used,grade_with_midterm,grade_without_midterm,grade_before_cap,final_course_grade,diagnostic_note
0,אדמוני,מאיה,322818386,mayaadm@post.bgu.ac.il,mayaadm@post.bgu.ac.il,1,7.368626,6.768626,0.6,,...,14.95,68.22,47.754,6.651,False,76.723626,77.858474,77.858474,77.858474,OK
1,אדר,נוי,323060202,noyad@post.bgu.ac.il,noyad@post.bgu.ac.il,0,3.204980,2.604980,0.6,,...,14.35,75.43,52.801,8.800,True,79.155980,78.173311,79.155980,79.155980,OK
2,אילביץ,אילנה,322772187,ilevich@post.bgu.ac.il,ilevich@post.bgu.ac.il,3,6.946500,6.346500,0.6,,...,12.95,47.52,33.264,6.501,False,NaN,NaN,47.520000,47.520000,Final exam below pass threshold; final grade i...
3,אלחואגרה,מחמד,325332419,alhwamoh@post.bgu.ac.il,alhwamoh@post.bgu.ac.il,0,2.850460,2.250460,0.6,,...,13.60,57.94,40.558,7.152,True,64.160460,63.342733,64.160460,64.160460,OK
4,אלקלעי,עדן,211340161,edenalk@post.bgu.ac.il,edenalk@post.bgu.ac.il,0,6.967248,6.367248,0.6,,...,14.75,83.90,58.730,9.002,True,89.449248,89.385831,89.449248,89.449248,OK


## Write Requested Outputs


In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if WRITE_MOODLE_IMPORT_CSV:
    upload_df.to_csv(MOODLE_IMPORT_CSV, index=False, encoding="utf-8-sig")
    print(f"Wrote compact Moodle import CSV: {MOODLE_IMPORT_CSV}")

if WRITE_DETAILED_AUDIT_CSV:
    audit_df.to_csv(DETAILED_AUDIT_CSV, index=False, encoding="utf-8-sig")
    print(f"Wrote detailed audit CSV: {DETAILED_AUDIT_CSV}")

if WRITE_GRADE_REPORT_MARKDOWN:
    report = "# Final Grade Report\n\n" + "\n\n".join(markdown_sections)
    GRADE_REPORT_MARKDOWN.write_text(report, encoding="utf-8")
    print(f"Wrote grade report markdown: {GRADE_REPORT_MARKDOWN}")


Wrote compact Moodle import CSV: generated_outputs\final_course_grade_moodle_import.csv
Wrote detailed audit CSV: generated_outputs\final_course_grade_audit.csv
Wrote grade report markdown: generated_outputs\final_course_grade_report.md


## Preview One Grade Report


In [10]:
if markdown_sections:
    print(markdown_sections[0])
else:
    print("No calculated grade reports to preview.")


## mayaadm@post.bgu.ac.il

### Course total

Final course grade: 77.86.
Final exam: 68.22; pass threshold 56; passed.
Review activity: 7.37/7.8.
Assignments: 14.95/15.
Midterm contribution: 6.65/10; midterm used: False.
Grade with midterm: 76.72.
Grade without midterm: 77.86.
Grade before cap: 77.86; cap 100.

### Tutor and WooClap

Tutor and WooClap review activity: 7.37/7.8 (raw 6.77, bonus 0.6).
S01: excused, WooClap 81.8%, Tutor 86.0%/on_time, combined 97.5%, max 0.60, total 0.58
S02: excused, WooClap 80.0%, Tutor 82.0%/on_time, combined 96.4%, max 0.60, total 0.58
S03: excused, WooClap 83.3%, Tutor 88.0%/on_time, combined 98.0%, max 0.60, total 0.59
S04: attended, WooClap 100.0%, Tutor 89.0%/on_time, combined 100.0%, max 0.60, total 0.60
S05: attended, WooClap 100.0%, Tutor 95.0%/on_time, combined 100.0%, max 0.60, total 0.60
S06: attended, WooClap 75.0%, Tutor 89.0%/on_time, combined 97.2%, max 0.60, total 0.58
S07: attended, WooClap 100.0%, Tutor 87.0%/on_time, combined 100.0%, 